# BERT-only Baseline — IndoBERT-base-p2

**Notebook:** `04_bert_only_baseline.ipynb`  
**Peneliti:** Khoeru Roziqin  
**Tanggal:** 03-05-26  

---

## Deskripsi

Notebook ini melatih **IndoBERT-base-p2 tanpa CNN** sebagai **DL baseline** pembanding terhadap proposed model (IndoBERT+CNN). Arsitektur hanya terdiri dari IndoBERT sebagai feature extractor diikuti Dense layer langsung ke output klasifikasi — tanpa komponen CNN apapun.

### Tujuan
Mengukur seberapa besar kontribusi CNN 1D dalam meningkatkan performa dibanding IndoBERT murni. Jika proposed model secara signifikan lebih baik, ini membuktikan nilai tambah arsitektur hybrid.

### Prinsip Fair Comparison
Untuk memastikan perbandingan yang **apple-to-apple**, baseline ini menggunakan:
- **BERT config identik** dengan config terbaik dari Phase 1 notebook 03
- **Kondisi data identik** dengan kondisi terbaik Phase 3 notebook 03 (S1)
- **Split data identik** — test set fixed yang sama (test_set_indices.npy)
- **Metrik evaluasi identik** — F1-Macro sebagai metrik utama

---

## Arsitektur BERT-only Baseline

```
Input Tweet (text_bert, max_len=128)
        ↓
IndoBERT-base-p2
  └─ [CLS] token → [batch, 768]
        ↓
Dropout(0.1)
        ↓
Dense(768 → 3) → Softmax [positive, negative, neutral]
```

**Perbedaan dengan Proposed Model:**
- ❌ Tidak ada CNN 1D multi-kernel
- ❌ Tidak ada GlobalMaxPool
- ❌ Tidak ada Dense(256) intermediate
- ✅ Menggunakan [CLS] token representation langsung ke classifier

---

## Konfigurasi Identik dengan Notebook 03

| Parameter | Nilai | Sumber |
|---|---|---|
| `lr_bert` | 1e-5 | Phase 1 terbaik notebook 03 |
| `batch_size` | 32 | Phase 1 terbaik notebook 03 |
| `weight_decay` | 0.01 | Fixed (non-bias/LayerNorm only) |
| `data_condition` | S1 | Phase 3 terbaik notebook 03 |
| `max_epochs` | 10 | Fixed Tier 2 |
| `patience` | 3 | Fixed Tier 2 |
| `warmup_ratio` | 0.1 | Fixed Tier 2 |
| `max_len` | 128 | Fixed Tier 2 |
| `n_folds` | 5 | Fixed Tier 2 |
| `seed` | 42 | Fixed Tier 2 |

**Referensi:**  
- Devlin et al. (2019). BERT. *NAACL-HLT 2019*.  
- Wilie et al. (2020). IndoNLU. *AACL-IJCNLP 2020*.

## 0. Setup

> Default: **Google Colab**. Untuk Kaggle: uncomment sel Kaggle, comment sel Colab.

In [ ]:
import sys, os

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')

if IS_KAGGLE:
    print('Environment: Kaggle Notebook')
elif IS_COLAB:
    print('Environment: Google Colab')
else:
    print('Environment: Local / Unknown')

In [ ]:
# ══════════════════════════════════════════════════════
#  SETUP — KAGGLE NOTEBOOK
# ══════════════════════════════════════════════════════

import os, torch

def check_gpu_kaggle():
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA tidak tersedia. Aktifkan GPU di Settings Kaggle.')
    try:
        t = torch.zeros(2, 2).cuda(); _ = t @ t; del t
        torch.cuda.empty_cache()
        cap = torch.cuda.get_device_capability()
        print(f'✅ GPU OK: {torch.cuda.get_device_name(0)} | sm_{cap[0]}{cap[1]}')
    except Exception as e:
        raise RuntimeError(
            f'GPU error: {e}\n'
            'Solusi: Settings → Accelerator → None → Save, aktifkan GPU kembali.'
        )
check_gpu_kaggle()
print('✅ Setup Kaggle selesai.')

In [ ]:
import time, warnings, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils import resample
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {DEVICE}')
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    print(f'Compute : sm_{cap[0]}{cap[1]}')
print(f'PyTorch : {torch.__version__}')

## 1. Konfigurasi Global

Semua config BERT identik dengan config terbaik notebook 03 untuk memastikan fair comparison.

In [ ]:
# ── PATH — KAGGLE ────────
INPUT_PATH      = '/kaggle/input/datasets/khoeruroziqin/final-mbg-labeled/final_mbg_labeled.csv'
NB03_OUTPUT_DIR = '/kaggle/working/model_results'
OUTPUT_DIR      = '/kaggle/working/baseline_results/bert_only'
MODEL_DIR       = '/kaggle/working/saved_models'
OUTPUTS_INPUT   = '/kaggle/input/datasets/khoeruroziqin/mbg-training-outputs'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR,  exist_ok=True)

# ── Restore file dari notebook 03 (Kaggle only) ───────────────────────────────
import shutil
_files_to_restore = [
    'test_set_indices.npy',
    'test_set_fixed.csv',
]
print('Restoring files dari mbg-training-outputs...')
for fname in _files_to_restore:
    src = f'{OUTPUTS_INPUT}/{fname}'
    dst = f'{NB03_OUTPUT_DIR}/{fname}'
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'  ✔ {fname}')
    else:
        print(f'  ⚠️  Tidak ditemukan: {fname}')
print('Done.')

# ── Model ─────────────────────────────────────────────────────────────────────
BERT_MODEL  = 'indobenchmark/indobert-base-p2'
MAX_LEN     = 128
NUM_CLASSES = 3
LABEL2ID    = {'positive': 0, 'negative': 1, 'neutral': 2}
ID2LABEL    = {0: 'positive', 1: 'negative', 2: 'neutral'}
LABEL_NAMES = ['positive', 'negative', 'neutral']

# ── Hyperparameter — identik dengan config terbaik notebook 03 ────────────────
# Sumber: Phase 1 terbaik (combo_id=2: lr_bert=1e-5, bs=32, wd=0.01)
#         Phase 3 terbaik: kondisi S1
LR_BERT      = 1e-5   # dari Phase 1 notebook 03
BATCH_SIZE   = 32     # dari Phase 1 notebook 03
WEIGHT_DECAY = 0.01   # fixed (non-bias/LayerNorm only)
DATA_COND    = 'S1'   # dari Phase 3 notebook 03

# ── Fixed Tier 2 ──────────────────────────────────────────────────────────────
MAX_EPOCHS   = 10
PATIENCE     = 3
N_FOLDS      = 5
WARMUP_RATIO = 0.1
ADAM_EPS     = 1e-8
DROPOUT_CLS  = 0.1   # dropout pada classifier head (standar BERT fine-tuning)

print('✅ Konfigurasi berhasil dimuat.')
print(f'   Model          : {BERT_MODEL}')
print(f'   lr_bert        : {LR_BERT}')
print(f'   batch_size     : {BATCH_SIZE}')
print(f'   weight_decay   : {WEIGHT_DECAY} (non-bias/LayerNorm only)')
print(f'   data_condition : {DATA_COND}')
print(f'   K-Fold         : {N_FOLDS}')
print(f'   max_epochs     : {MAX_EPOCHS} (early stopping patience={PATIENCE})')
print(f'   Arsitektur     : IndoBERT → [CLS] → Dropout({DROPOUT_CLS}) → Dense(3)')

## 2. Load & Split Data

Menggunakan **test_set_indices.npy yang sama** dengan notebook 03 untuk memastikan test set identik.

In [ ]:
print('Memuat data berlabel...')
df = pd.read_csv(INPUT_PATH, low_memory=False)
df = df[df['label'].isin(['positive','negative','neutral'])].copy()
df['label_id'] = df['label'].map(LABEL2ID)
df = df.dropna(subset=['text_bert','label_id']).reset_index(drop=True)

print(f'Total data berlabel: {len(df):,}')
for lbl, cnt in df['label'].value_counts().items():
    print(f'  {lbl:12s}: {cnt:,} ({cnt/len(df)*100:.1f}%)')

# Load test set indices dari notebook 03 — WAJIB sama persis
test_idx_path = f'{NB03_OUTPUT_DIR}/test_set_indices.npy'
assert os.path.exists(test_idx_path), (
    f'❌ test_set_indices.npy tidak ditemukan di {test_idx_path}!\n'
    'Pastikan file output notebook 03 sudah tersedia.'
)

test_indices = np.load(test_idx_path)
train_mask   = np.ones(len(df), dtype=bool)
train_mask[test_indices] = False
df_trainval  = df[train_mask].reset_index(drop=True)
df_test      = df.iloc[test_indices].reset_index(drop=True)

print(f'\nTest set index dimuat dari notebook 03 (identik).')
print(f'Train+Val : {len(df_trainval):,} | Test (FIXED): {len(df_test):,}')
print(f'\nDistribusi test set:')
for lbl, cnt in df_test['label'].value_counts().items():
    print(f'  {lbl:12s}: {cnt:,} ({cnt/len(df_test)*100:.1f}%)')

## 3. Dataset Class & Arsitektur BERT-only

Arsitektur **jauh lebih sederhana** dari proposed model:
- Menggunakan representasi **[CLS] token** dari output BERT (bukan sequence hidden states)
- Langsung ke classifier tanpa CNN, pooling, atau Dense intermediate
- Ini adalah arsitektur fine-tuning BERT standar (Devlin et al., 2019)

In [ ]:
print(f'Memuat tokenizer: {BERT_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
print('✅ Tokenizer berhasil dimuat.')


class MBGDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts, self.labels         = texts, labels
        self.tokenizer, self.max_len    = tokenizer, max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]), max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids'     : enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label'         : torch.tensor(self.labels[idx], dtype=torch.long)
        }


class IndoBERTOnly(nn.Module):
    """
    DL Baseline: IndoBERT-base-p2 dengan classifier head standar.
    Menggunakan [CLS] token representation → Dropout → Dense(3).

    Ini adalah arsitektur fine-tuning BERT standar (Devlin et al., 2019)
    tanpa modifikasi arsitektur apapun — berbeda dengan proposed model
    yang menambahkan CNN 1D multi-kernel di atasnya.

    Perbandingan dengan proposed model:
      Proposed : BERT → seq_output → CNN → GlobalMaxPool → Dense(256) → Dense(3)
      Baseline : BERT → [CLS]      → Dropout → Dense(3)
    """
    def __init__(self, bert_model_name, num_classes, dropout=0.1):
        super().__init__()
        self.bert       = AutoModel.from_pretrained(bert_model_name)
        hidden          = self.bert.config.hidden_size  # 768
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, num_classes)

    def forward(self, input_ids, attention_mask):
        # [CLS] token: posisi 0 dari pooler_output atau last_hidden_state[:,0,:]
        output   = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_repr = output.last_hidden_state[:, 0, :]  # [batch, 768]
        cls_repr = self.dropout(cls_repr)
        return self.classifier(cls_repr)              # [batch, num_classes]


print('✅ Dataset class dan arsitektur IndoBERTOnly berhasil didefinisikan.')
print('   Arsitektur: IndoBERT → [CLS] token → Dropout(0.1) → Dense(3)')
print('   Tanpa CNN, tanpa GlobalMaxPool, tanpa Dense(256) intermediate')

## 4. Training Utilities

Menggunakan fungsi yang sama dengan notebook 03, dengan satu perbedaan:
- `build_optimizer` hanya memiliki **2 parameter group** (bukan 4) karena tidak ada komponen CNN
- Weight decay exclusion tetap diterapkan pada bias dan LayerNorm

In [ ]:
# ── Metrik ────────────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred):
    return {
        'accuracy'   : round(accuracy_score(y_true, y_pred), 4),
        'precision'  : round(precision_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'recall'     : round(recall_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'f1_macro'   : round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'f1_weighted': round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 4),
    }


# ── Train & Eval Loop ─────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, preds_all, labels_all = 0.0, [], []
    for batch in loader:
        ids, mask, lbls = (batch['input_ids'].to(device),
                           batch['attention_mask'].to(device),
                           batch['label'].to(device))
        optimizer.zero_grad()
        logits = model(ids, mask)
        loss   = criterion(logits, lbls)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step(); scheduler.step()
        total_loss += loss.item()
        preds_all.extend(torch.argmax(logits, 1).cpu().numpy())
        labels_all.extend(lbls.cpu().numpy())
    return total_loss / len(loader), f1_score(labels_all, preds_all, average='macro', zero_division=0)


def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, preds_all, labels_all, probs_all = 0.0, [], [], []
    with torch.no_grad():
        for batch in loader:
            ids, mask, lbls = (batch['input_ids'].to(device),
                               batch['attention_mask'].to(device),
                               batch['label'].to(device))
            logits = model(ids, mask)
            total_loss += criterion(logits, lbls).item()
            preds_all.extend(torch.argmax(logits, 1).cpu().numpy())
            labels_all.extend(lbls.cpu().numpy())
            probs_all.append(torch.softmax(logits, dim=1).cpu().numpy())
    probs = np.vstack(probs_all)
    return total_loss / len(loader), compute_metrics(labels_all, preds_all), preds_all, labels_all, probs


# ── Penanganan Imbalanced ──────────────────────────────────────────────────────
def apply_undersampling(X, y, seed=SEED):
    min_n = min(np.bincount(y.astype(int)))
    X_res, y_res = [], []
    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        s   = resample(idx, n_samples=min_n, random_state=seed, replace=False)
        X_res.append(X[s]); y_res.append(y[s])
    X_res, y_res = np.concatenate(X_res), np.concatenate(y_res)
    shuf = np.random.RandomState(seed).permutation(len(X_res))
    return X_res[shuf], y_res[shuf]


def build_criterion(y_train, condition, device):
    if condition == 'S2':
        return nn.CrossEntropyLoss()
    n_samples = len(y_train)
    weights   = np.zeros(NUM_CLASSES, dtype=np.float32)
    for cls in range(NUM_CLASSES):
        n_cls        = np.sum(y_train == cls)
        weights[cls] = (n_samples / (NUM_CLASSES * n_cls)) if n_cls > 0 else 0.0
    return nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float).to(device))


def build_optimizer(model, lr_bert, weight_decay):
    """
    2 parameter groups untuk BERT-only (tanpa CNN):
      Group 1: BERT weight (non-bias/LayerNorm) → lr_bert, weight_decay
      Group 2: BERT bias & LayerNorm            → lr_bert, weight_decay=0.0
    Ref: Loshchilov & Hutter (2019) — AdamW decoupled weight decay
    """
    no_decay     = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']
    decay_params = [
        p for n, p in model.named_parameters()
        if not any(nd in n for nd in no_decay) and p.requires_grad
    ]
    nodecay_params = [
        p for n, p in model.named_parameters()
        if any(nd in n for nd in no_decay) and p.requires_grad
    ]
    param_groups = [
        {'params': decay_params,   'lr': lr_bert, 'weight_decay': weight_decay},
        {'params': nodecay_params, 'lr': lr_bert, 'weight_decay': 0.0},
    ]
    n_total   = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_grouped = sum(p.numel() for g in param_groups for p in g['params'])
    assert n_total == n_grouped, f'Param mismatch: {n_total} vs {n_grouped}'
    return AdamW(param_groups, eps=ADAM_EPS)


print('✅ Training utilities berhasil didefinisikan.')
print('   Optimizer: AdamW 2 param group (weight decay exclusion pada bias & LayerNorm)')

## 5. Stratified K-Fold Training

K-Fold 5 pada 5.313 data train+val dengan kondisi S1 (Class Weighting) — identik dengan Phase 3 notebook 03.

> ⏱️ Estimasi waktu: **~3–4 jam** GPU T4 (5 fold × ~40 menit per fold).

In [ ]:
print(f'Menjalankan K-Fold training — IndoBERT-only Baseline')
print(f'Kondisi data : {DATA_COND} (identik dengan kondisi terbaik notebook 03)')
print(f'Config BERT  : lr={LR_BERT} | bs={BATCH_SIZE} | wd={WEIGHT_DECAY}')
print(f'K-Fold       : {N_FOLDS} | max_epochs={MAX_EPOCHS} | patience={PATIENCE}')

skf          = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
X_tv         = df_trainval['text_bert'].values
y_tv         = df_trainval['label_id'].values
fold_metrics = []
fold_details = []
epoch_history= []
t_start_total= time.time()

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_tv, y_tv), 1):
    print(f'\n{"─"*55}\nFold {fold}/{N_FOLDS}\n{"─"*55}')

    X_tr, y_tr   = X_tv[tr_idx], y_tv[tr_idx]
    X_val, y_val = X_tv[val_idx], y_tv[val_idx]

    if DATA_COND == 'S2':
        X_tr, y_tr = apply_undersampling(X_tr, y_tr)

    criterion = build_criterion(y_tr, DATA_COND, DEVICE)

    # num_workers: 2 Colab | 0 Kaggle
    train_dl = DataLoader(MBGDataset(X_tr, y_tr, tokenizer, MAX_LEN),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_dl   = DataLoader(MBGDataset(X_val, y_val, tokenizer, MAX_LEN),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model        = IndoBERTOnly(BERT_MODEL, NUM_CLASSES, DROPOUT_CLS).to(DEVICE)
    optimizer    = build_optimizer(model, LR_BERT, WEIGHT_DECAY)
    total_steps  = len(train_dl) * MAX_EPOCHS
    scheduler    = get_linear_schedule_with_warmup(
        optimizer, int(total_steps * WARMUP_RATIO), total_steps)

    best_val_loss = float('inf')
    patience_cnt  = 0
    best_metrics  = None
    best_epoch    = 0

    epoch_bar = tqdm(range(1, MAX_EPOCHS + 1),
                     desc=f'  Fold {fold}/{N_FOLDS}', leave=False,
                     bar_format='{l_bar}{bar:20}{r_bar}')

    for epoch in epoch_bar:
        tr_loss, tr_f1       = train_epoch(model, train_dl, optimizer, scheduler, criterion, DEVICE)
        val_loss, m, _, _, _ = eval_epoch(model, val_dl, criterion, DEVICE)

        epoch_history.append({
            'fold'         : fold,
            'epoch'        : epoch,
            'train_loss'   : round(tr_loss, 4),
            'val_loss'     : round(val_loss, 4),
            'train_f1'     : round(tr_f1, 4),
            **{f'val_{k}': v for k, v in m.items()},
            'is_best_epoch': False
        })

        epoch_bar.set_postfix({
            'tr_loss': f'{tr_loss:.4f}', 'val_loss': f'{val_loss:.4f}',
            'f1': f'{m["f1_macro"]:.4f}', 'pat': f'{patience_cnt}/{PATIENCE}'
        })

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_metrics  = m
            patience_cnt  = 0
            best_epoch    = epoch
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                epoch_bar.set_description(f'  Fold {fold}/{N_FOLDS} [stop ep{epoch}]')
                break

    # Tandai best epoch
    for rec in epoch_history:
        if rec['fold'] == fold and rec['epoch'] == best_epoch:
            rec['is_best_epoch'] = True

    fold_details.append({
        'fold'         : fold,
        'best_epoch'   : best_epoch,
        'best_val_loss': round(best_val_loss, 4),
        **{f'val_{k}': v for k, v in best_metrics.items()}
    })
    fold_metrics.append(best_metrics)

    elapsed = (time.time() - t_start_total) / 60
    print(f'  ├─ Fold {fold}/{N_FOLDS} → F1={best_metrics["f1_macro"]:.4f} | '
          f'Acc={best_metrics["accuracy"]:.4f} | best_ep={best_epoch}')
    print(f'  └─ Elapsed: {elapsed:.1f} menit')

    del model; torch.cuda.empty_cache()

# ── Rata-rata K-Fold ──────────────────────────────────────────────────────────
keys     = fold_metrics[0].keys()
avg      = {k: round(np.mean([m[k] for m in fold_metrics]), 4) for k in keys}
std      = {f'{k}_std': round(np.std([m[k] for m in fold_metrics]), 4) for k in keys}
kfold_results = {**avg, **std}

# Simpan
pd.DataFrame(epoch_history).to_csv(f'{OUTPUT_DIR}/bert_only_epoch_history.csv', index=False)
pd.DataFrame(fold_details).to_csv(f'{OUTPUT_DIR}/bert_only_fold_details.csv', index=False)

total_time = time.time() - t_start_total
print(f'\n{"="*55}')
print(f' K-FOLD SELESAI — {total_time/3600:.1f} jam')
print(f'{"="*55}')
print(f'  F1-Macro     : {avg["f1_macro"]:.4f} ± {std["f1_macro_std"]:.4f}')
print(f'  F1-Weighted  : {avg["f1_weighted"]:.4f} ± {std["f1_weighted_std"]:.4f}')
print(f'  Accuracy     : {avg["accuracy"]:.4f} ± {std["accuracy_std"]:.4f}')
print(f'  Precision    : {avg["precision"]:.4f} ± {std["precision_std"]:.4f}')
print(f'  Recall       : {avg["recall"]:.4f} ± {std["recall_std"]:.4f}')
print(f'\nDetail per fold:')
for fd in fold_details:
    print(f'  Fold {fd["fold"]}: F1={fd["val_f1_macro"]:.4f} | Acc={fd["val_accuracy"]:.4f} | best_ep={fd["best_epoch"]}')

## 6. Visualisasi K-Fold

In [ ]:
df_eh = pd.DataFrame(epoch_history)
df_fd = pd.DataFrame(fold_details)
colors = ['#534AB7','#E24B4A','#1D9E75','#EF9F27','#AFA9EC']

# ── Learning Curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for fi, (fold_num, grp) in enumerate(df_eh.groupby('fold')):
    c = colors[fi]
    best_ep = grp[grp['is_best_epoch']]['epoch'].values
    axes[0].plot(grp['epoch'], grp['train_loss'], '--', color=c, alpha=0.5, linewidth=1)
    axes[0].plot(grp['epoch'], grp['val_loss'],   '-',  color=c, alpha=0.9, linewidth=1.5,
                 label=f'Fold {fold_num}')
    if len(best_ep):
        bp = grp[grp['epoch'] == best_ep[0]]
        axes[0].scatter(best_ep[0], bp['val_loss'].values[0], color=c, s=60, zorder=5)
    axes[1].plot(grp['epoch'], grp['val_f1_macro'], '-', color=c, linewidth=1.5,
                 label=f'Fold {fold_num}')

axes[0].set_title('Loss Curves (train=dashed, val=solid)\nIndoBERT-only Baseline', fontsize=11)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=8); sns.despine(ax=axes[0])

axes[1].set_title('Validation F1-Macro per Epoch\nIndoBERT-only Baseline', fontsize=11)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1-Macro')
axes[1].legend(fontsize=8); sns.despine(ax=axes[1])

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/bert_only_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Stabilitas K-Fold ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
metrics_box = ['val_f1_macro','val_accuracy','val_precision','val_recall']
labels_box  = ['F1-Macro','Accuracy','Precision','Recall']
data_box    = [df_fd[m].values for m in metrics_box]
bp          = ax.boxplot(data_box, patch_artist=True, widths=0.5)
for patch in bp['boxes']:
    patch.set_facecolor('#534AB7'); patch.set_alpha(0.7)
for median in bp['medians']:
    median.set_color('white'); median.set_linewidth(2)
ax.set_xticklabels(labels_box, fontsize=10)
ax.set_ylabel('Score per Fold', fontsize=11)
ax.set_title('Stabilitas K-Fold — IndoBERT-only Baseline', fontsize=12)
ax.set_ylim(0.75, 1.0)
sns.despine(ax=ax); plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/bert_only_kfold_stability.png', dpi=150, bbox_inches='tight')
plt.show()
print('✔ Visualisasi K-Fold tersimpan.')

## 7. Final Training & Evaluasi Test Set

Model final dilatih pada **seluruh 5.313 train+val**, dievaluasi pada **test set fixed 1.329 data** — identik dengan evaluasi final notebook 03.

In [ ]:
print(f'{"="*55}\n FINAL TRAINING & EVALUASI TEST SET\n{"="*55}')
print(f' Arsitektur  : IndoBERT → [CLS] → Dropout({DROPOUT_CLS}) → Dense(3)')
print(f' Config      : lr={LR_BERT} | bs={BATCH_SIZE} | wd={WEIGHT_DECAY}')
print(f' Kondisi     : {DATA_COND} | Train+Val={len(df_trainval):,} | Test={len(df_test):,}')

X_full = df_trainval['text_bert'].values
y_full = df_trainval['label_id'].values
if DATA_COND == 'S2':
    X_full, y_full = apply_undersampling(X_full, y_full)

criterion_f = build_criterion(y_full, DATA_COND, DEVICE)
# num_workers: 2 Colab | 0 Kaggle
train_dl_f  = DataLoader(MBGDataset(X_full, y_full, tokenizer, MAX_LEN),
                         batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_dl_f   = DataLoader(MBGDataset(df_test['text_bert'].values,
                                    df_test['label_id'].values, tokenizer, MAX_LEN),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model_f     = IndoBERTOnly(BERT_MODEL, NUM_CLASSES, DROPOUT_CLS).to(DEVICE)
optimizer_f = build_optimizer(model_f, LR_BERT, WEIGHT_DECAY)
total_steps = len(train_dl_f) * MAX_EPOCHS
scheduler_f = get_linear_schedule_with_warmup(
    optimizer_f, int(total_steps * WARMUP_RATIO), total_steps)

final_epoch_hist = []
best_loss, patience_cnt, best_state = float('inf'), 0, None
t_train = time.time()

epoch_bar = tqdm(range(1, MAX_EPOCHS+1), desc='Final Training',
                 bar_format='{l_bar}{bar:25}{r_bar}')
for epoch in epoch_bar:
    tr_loss, tr_f1       = train_epoch(model_f, train_dl_f, optimizer_f, scheduler_f, criterion_f, DEVICE)
    val_loss, m, _, _, _ = eval_epoch(model_f, test_dl_f, criterion_f, DEVICE)

    final_epoch_hist.append({
        'epoch': epoch, 'train_loss': round(tr_loss,4),
        'val_loss': round(val_loss,4), 'train_f1': round(tr_f1,4),
        **{f'val_{k}': v for k, v in m.items()}
    })
    epoch_bar.set_postfix({'tr_loss': f'{tr_loss:.4f}', 'val_loss': f'{val_loss:.4f}',
                           'f1': f'{m["f1_macro"]:.4f}', 'pat': f'{patience_cnt}/{PATIENCE}'})
    if val_loss < best_loss:
        best_loss    = val_loss
        best_state   = {k: v.clone() for k, v in model_f.state_dict().items()}
        patience_cnt = 0
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'  Early stopping ep{epoch}'); break

pd.DataFrame(final_epoch_hist).to_csv(
    f'{OUTPUT_DIR}/bert_only_final_epoch_history.csv', index=False)
print(f' Training selesai — {(time.time()-t_train)/60:.1f} menit')

# ── Evaluasi final ────────────────────────────────────────────────────────────
model_f.load_state_dict(best_state)
_, final_metrics, all_preds, all_labels, all_probs = eval_epoch(
    model_f, test_dl_f, criterion_f, DEVICE)

per_class = f1_score(all_labels, all_preds, average=None, zero_division=0)

print(f'\n{"="*55}\n EVALUASI FINAL — TEST SET (IndoBERT-only)\n{"="*55}')
for k, v in final_metrics.items(): print(f'  {k:15s}: {v:.4f}')
print(f'\n{classification_report(all_labels, all_preds, target_names=LABEL_NAMES)}')
print(f' Per-class F1: ' + ' | '.join(
    [f'{LABEL_NAMES[i]}={per_class[i]:.4f}' for i in range(NUM_CLASSES)]))

# ── Confusion matrix pair ─────────────────────────────────────────────────────
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, data, fmt, title in [
    (axes[0], cm,      'd',   f'Confusion Matrix (count)\nIndoBERT-only | F1-Macro={final_metrics["f1_macro"]:.4f}'),
    (axes[1], cm_norm, '.2%', f'Confusion Matrix (normalized %)\nIndoBERT-only | Accuracy={final_metrics["accuracy"]:.4f}')
]:
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/bert_only_cm_pair.png', dpi=150, bbox_inches='tight')
plt.show()
print('  ✔ Confusion matrix pair disimpan.')

# ── Test predictions CSV ──────────────────────────────────────────────────────
df_pred = df_test[['full_text','text_bert','label']].copy() \
          if 'full_text' in df_test.columns \
          else df_test[['text_bert','label']].copy()
df_pred['predicted']     = [ID2LABEL[p] for p in all_preds]
df_pred['correct']       = df_pred['label'] == df_pred['predicted']
df_pred['confidence']    = np.round(np.max(all_probs, axis=1), 4)
df_pred['prob_positive'] = np.round(all_probs[:, 0], 4)
df_pred['prob_negative'] = np.round(all_probs[:, 1], 4)
df_pred['prob_neutral']  = np.round(all_probs[:, 2], 4)
df_pred.to_csv(f'{OUTPUT_DIR}/bert_only_test_predictions.csv',
               index=False, encoding='utf-8-sig')
print(f'  ✔ bert_only_test_predictions.csv ({len(df_pred):,} baris)')
print(f'    Accuracy: {df_pred["correct"].mean()*100:.2f}%')

# ── Simpan model ──────────────────────────────────────────────────────────────
torch.save({
    'model_state_dict': model_f.state_dict(),
    'config'          : {
        'lr_bert': LR_BERT, 'batch_size': BATCH_SIZE,
        'weight_decay': WEIGHT_DECAY, 'dropout_cls': DROPOUT_CLS,
        'data_condition': DATA_COND,
        'no_decay': ['bias','LayerNorm.weight','LayerNorm.bias']
    },
    'fixed_config'    : {
        'max_len': MAX_LEN, 'warmup_ratio': WARMUP_RATIO,
        'adam_eps': ADAM_EPS, 'seed': SEED
    },
    'test_metrics'    : final_metrics,
    'per_class_f1'    : {LABEL_NAMES[i]: round(per_class[i], 4) for i in range(NUM_CLASSES)},
    'kfold_metrics'   : kfold_results,
    'label2id': LABEL2ID, 'id2label': ID2LABEL, 'bert_model_name': BERT_MODEL,
}, f'{MODEL_DIR}/indobert_only_final.pt')
print(f'  ✔ Model disimpan: indobert_only_final.pt')

## 8. Ringkasan & Perbandingan Awal dengan Proposed Model

In [ ]:
# ── Simpan hasil ke CSV untuk notebook 06 (model comparison) ─────────────────
bert_only_summary = {
    'model'         : 'IndoBERT-only (Baseline DL)',
    'architecture'  : 'IndoBERT → [CLS] → Dropout(0.1) → Dense(3)',
    'data_condition': DATA_COND,
    'test_accuracy' : final_metrics['accuracy'],
    'test_precision': final_metrics['precision'],
    'test_recall'   : final_metrics['recall'],
    'test_f1_macro' : final_metrics['f1_macro'],
    'test_f1_weighted': final_metrics['f1_weighted'],
    'f1_positive'   : round(per_class[0], 4),
    'f1_negative'   : round(per_class[1], 4),
    'f1_neutral'    : round(per_class[2], 4),
    'kfold_f1_macro_mean': kfold_results['f1_macro'],
    'kfold_f1_macro_std' : kfold_results['f1_macro_std'],
    'lr_bert'       : LR_BERT,
    'batch_size'    : BATCH_SIZE,
    'weight_decay'  : WEIGHT_DECAY,
}
pd.DataFrame([bert_only_summary]).to_csv(
    f'{OUTPUT_DIR}/bert_only_summary.csv', index=False)

# ── Perbandingan awal dengan proposed model ───────────────────────────────────
# Hasil proposed model dari notebook 03
proposed_f1    = 0.8453  # update jika berbeda
proposed_acc   = 0.8495
bert_only_f1   = final_metrics['f1_macro']
bert_only_acc  = final_metrics['accuracy']
delta_f1       = round(proposed_f1 - bert_only_f1, 4)
delta_acc      = round(proposed_acc - bert_only_acc, 4)

print(f'{"="*60}')
print(f' RINGKASAN — IndoBERT-only Baseline')
print(f'{"="*60}')
print(f'  Arsitektur     : IndoBERT → [CLS] → Dropout → Dense(3)')
print(f'  Config BERT    : lr={LR_BERT} | bs={BATCH_SIZE} | wd={WEIGHT_DECAY}')
print(f'  Kondisi data   : {DATA_COND} (class weighting)')
print(f'{"─"*60}')
print(f'  K-Fold F1-Macro: {kfold_results["f1_macro"]:.4f} ± {kfold_results["f1_macro_std"]:.4f}')
print(f'  Test F1-Macro  : {bert_only_f1:.4f}')
print(f'  Test Accuracy  : {bert_only_acc:.4f}')
print(f'  Per-class F1   : pos={per_class[0]:.4f} | neg={per_class[1]:.4f} | neu={per_class[2]:.4f}')
print(f'{"─"*60}')
print(f' Perbandingan awal vs Proposed Model (IndoBERT+CNN):')
print(f'  Proposed F1-Macro : {proposed_f1:.4f}')
print(f'  Baseline F1-Macro : {bert_only_f1:.4f}')
print(f'  Delta F1-Macro    : {delta_f1:+.4f} {"(proposed lebih baik ✅)" if delta_f1 > 0 else "(baseline lebih baik ⚠️)"}')
print(f'  Delta Accuracy    : {delta_acc:+.4f}')
print(f'{"="*60}')
print(f'\n Output yang dihasilkan:')
outputs = [
    ('bert_only_epoch_history.csv',       'Epoch history K-Fold'),
    ('bert_only_fold_details.csv',        'Metrik terbaik per fold'),
    ('bert_only_final_epoch_history.csv', 'Epoch history final training'),
    ('bert_only_learning_curves.png',     'Learning curves K-Fold'),
    ('bert_only_kfold_stability.png',     'Box plot stabilitas K-Fold'),
    ('bert_only_cm_pair.png',             'Confusion matrix unnormalized + normalized'),
    ('bert_only_test_predictions.csv',    'Prediksi per tweet + confidence'),
    ('bert_only_summary.csv',            'Ringkasan metrik untuk notebook 06'),
    ('indobert_only_final.pt',           'Model checkpoint final'),
]
for fname, desc in outputs:
    path   = f'{OUTPUT_DIR}/{fname}' if not fname.endswith('.pt') else f'{MODEL_DIR}/{fname}'
    exists = '✔' if os.path.exists(path) else '○'
    print(f'  {exists} {fname:<45} {desc}')
print(f'\n Lanjut ke Notebook 05 — ML Baseline (SVM, LR, RF)')

---

## Catatan Metodologis untuk Laporan

### Arsitektur BERT-only Baseline
IndoBERT-only menggunakan representasi **[CLS] token** dari layer terakhir BERT sebagai input classifier. Representasi [CLS] dirancang oleh Devlin et al. (2019) sebagai agregasi seluruh konteks sekuens untuk task klasifikasi. Ini berbeda dengan proposed model yang menggunakan **seluruh sequence hidden states** sebagai input CNN untuk menangkap pola n-gram lokal.

### Prinsip Fair Comparison
Untuk memastikan perbandingan yang valid, baseline ini menggunakan konfigurasi BERT yang **identik** dengan proposed model: learning rate, batch size, weight decay, weight decay exclusion, kondisi data, dan split test set. Satu-satunya perbedaan adalah ada/tidaknya komponen CNN — sehingga delta performa yang terukur dapat secara langsung diatribusikan pada kontribusi CNN.

### Interpretasi Hasil
Jika proposed model (IndoBERT+CNN) memberikan F1-Macro yang lebih tinggi dari baseline ini, ini membuktikan bahwa CNN 1D multi-kernel memberikan nilai tambah dalam menangkap fitur sentimen lokal yang tidak tertangkap oleh representasi [CLS] BERT saja. Sebaliknya, jika hasilnya setara, ini menunjukkan bahwa BERT sudah cukup kuat untuk task ini tanpa modifikasi arsitektur.

**Referensi:**
- Devlin et al. (2019). BERT. *NAACL-HLT 2019*.
- Loshchilov & Hutter (2019). AdamW. *ICLR 2019*.
- Wilie et al. (2020). IndoNLU. *AACL-IJCNLP 2020*.